# Data Analyst Market Intelligence - Data Cleaning & Preprocessing

## Goal
Clean and prepare the Indian job-market dataset for the Data Analyst Job Market Tracker.

**Workflow:** Raw Excel → Python cleaning → Clean dataset → MySQL → Power BI

Python is used mainly for understanding the data, checking quality, cleaning salary values, validating experience, preparing skills, and identifying Data Analyst jobs. Detailed business analysis will be done later in SQL.


## 1. Import libraries and load the dataset

In [2]:
import pandas as pd
import numpy as np


df = pd.read_csv("indian-job-market-dataset-2025.csv")

df.head()


,title,jobId,currency,jobUploaded,companyName,tagsAndSkills,experience,salary,location,companyId,ReviewsCount,AggregateRating,jobDescription,minimumSalary,maximumSalary,minimumExperience,maximumExperience
0,Sr. HR Recruiter (NON IT),270925008041,INR,6 Days Ago,Orion,"Communication,Manpower,Staffing,Convincing Pow...",2-4 Yrs,2-4 Lacs PA,Kolkata(Chinar Park),645563,NaN,NaN,Preferred candidate profile . .,200000.0,400000.0,2.0,4.0
1,Fire And Safety Officer,270925007584,INR,6 Days Ago,"Apollo Hospitals International Limited, Ahmedabad","Safety Officer Activities,Fire Protection,Fire...",6-11 Yrs,3-5 Lacs PA,"Gandhinagar, Ahmedabad",14072,5162.0,4.0,"Ensure active Fire Protection System,such as F...",300000.0,500000.0,6.0,11.0
2,Opening For Performance Marketing - Chennai,270925007492,INR,6 Days Ago,TVS Credit Services Ltd,"Performance Marketing,User Acquisition,growth ...",12-18 Yrs,Not disclosed,Chennai,1324750,2892.0,4.2,MBA Marketing (preferred Tier II or III B- Sch...,0.0,0.0,12.0,18.0
3,Medical Billing Executive,270925007443,INR,6 Days Ago,GNR Global Services,"Fluent English,Spoken English,Good English Com...",0-3 Yrs,"70,000-2 Lacs PA","Mohali, Chandigarh, Kharar, Zirakpur",123804403,NaN,NaN,Job Title-Medical Billing Executive\nLocation-...,70000.0,200000.0,0.0,3.0
4,Senior Group Product Manager - CNS Therapy,270925007430,INR,6 Days Ago,Cadila Pharmaceuticals,"Product Marketing,CNS,Product Management,Nephr...",5-10 Yrs,8-18 Lacs PA,Ahmedabad,14957,2134.0,3.4,Principal Tasks & Responsibilities : (Please w...,800000.0,1800000.0,5.0,10.0


## 2. Understand the dataset

In [3]:
df.shape


(97929, 17)

In [4]:
df.columns


Index(['title', 'jobId', 'currency', 'jobUploaded', 'companyName',
       'tagsAndSkills', 'experience', 'salary', 'location', 'companyId',
       'ReviewsCount', 'AggregateRating', 'jobDescription', 'minimumSalary',
       'maximumSalary', 'minimumExperience', 'maximumExperience'],
      dtype='object')

In [5]:
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 97929 entries, 0 to 97928
Data columns (total 17 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   title              97929 non-null  object 
 1   jobId              97929 non-null  int64  
 2   currency           97929 non-null  object 
 3   jobUploaded        97929 non-null  object 
 4   companyName        97925 non-null  object 
 5   tagsAndSkills      97358 non-null  object 
 6   experience         95824 non-null  object 
 7   salary             97929 non-null  object 
 8   location           97929 non-null  object 
 9   companyId          97929 non-null  int64  
 10  ReviewsCount       62677 non-null  float64
 11  AggregateRating    62677 non-null  float64
 12  jobDescription     97929 non-null  object 
 13  minimumSalary      97358 non-null  float64
 14  maximumSalary      97358 non-null  float64
 15  minimumExperience  97358 non-null  float64
 16  maximumExperience  973

## 3. Check missing values

In [6]:
missing = pd.DataFrame({
    "missing_count": df.isnull().sum(),
    "missing_percentage": (df.isnull().sum() / len(df) * 100).round(2)
})

missing.sort_values("missing_count", ascending=False)


,missing_count,missing_percentage
ReviewsCount,35252,36.00
AggregateRating,35252,36.00
experience,2105,2.15
maximumExperience,571,0.58
minimumExperience,571,0.58
maximumSalary,571,0.58
tagsAndSkills,571,0.58
minimumSalary,571,0.58
companyName,4,0.00
jobDescription,0,0.00


## 4. Check duplicate records

In [7]:
duplicate_rows = df.duplicated().sum()
duplicate_job_ids = df["jobId"].duplicated().sum()

print("Duplicate rows:", duplicate_rows)
print("Duplicate job IDs:", duplicate_job_ids)


Duplicate rows: 247
Duplicate job IDs: 250


## 5. Remove exact duplicate rows

Exact duplicate rows do not add new information, so they are removed. The `jobId` check is kept separate because duplicate IDs may need further investigation later.


In [8]:
df_clean = df.drop_duplicates().copy()

print("Rows before:", len(df))
print("Rows after:", len(df_clean))


Rows before: 97929
Rows after: 97682


## 6. Check important fields

In [9]:
print("Currency:")
print(df_clean["currency"].value_counts(dropna=False))

print("\nSalary:")
print(df_clean["salary"].value_counts(dropna=False).head(20))

print("\nExperience:")
print(df_clean["experience"].value_counts(dropna=False).head(20))


Currency:
currency
INR    97553
USD      129
Name: count, dtype: int64

Salary:
salary
Not disclosed       63920
3-5 Lacs PA           584
3-6 Lacs PA           549
3-4 Lacs PA           494
2-3 Lacs PA           470
2-4 Lacs PA           434
4-6 Lacs PA           434
10-20 Lacs PA         380
15-25 Lacs PA         378
3-4.5 Lacs PA         373
2.5-4 Lacs PA         353
2.5-3.5 Lacs PA       338
Unpaid                333
2-3.5 Lacs PA         290
3-3.5 Lacs PA         267
3-8 Lacs PA           266
50,000-3 Lacs PA      265
2-2.5 Lacs PA         261
2-5 Lacs PA           258
1-3 Lacs PA           252
Name: count, dtype: int64

Experience:
experience
5-10 Yrs    6433
3-8 Yrs     5746
1-3 Yrs     3904
2-5 Yrs     3767
1-6 Yrs     3620
2-7 Yrs     3606
0-5 Yrs     3286
3-5 Yrs     3283
5-8 Yrs     3264
4-9 Yrs     2868
1-5 Yrs     2860
0-1 Yrs     2579
1-4 Yrs     2418
6-11 Yrs    2379
0-2 Yrs     2299
2-4 Yrs     2262
7-12 Yrs    2196
8-13 Yrs    2127
3-6 Yrs     2124
NaN         2102
Nam

## 7. Clean salary data

The `salary` column contains annual ranges, monthly salary, `Not disclosed`, and `Unpaid`.

We create three comparable salary columns:

- `salary_min_lpa`
- `salary_max_lpa`
- `salary_mid_lpa`

LPA means **Lakhs Per Annum**.


In [10]:
df_clean["salary_min_lpa"] = np.nan
df_clean["salary_max_lpa"] = np.nan
df_clean["salary_mid_lpa"] = np.nan


### 7.1 Monthly salaries

In [11]:
monthly_mask = df_clean["salary"].str.contains(
    "/month",
    case=False,
    na=False
)

monthly_salary = pd.to_numeric(
    df_clean.loc[monthly_mask, "salary"]
    .str.extract(r"([\d,]+)")[0]
    .str.replace(",", ""),
    errors="coerce"
)

monthly_salary_lpa = (monthly_salary * 12) / 100000

df_clean.loc[monthly_mask, "salary_min_lpa"] = monthly_salary_lpa
df_clean.loc[monthly_mask, "salary_max_lpa"] = monthly_salary_lpa
df_clean.loc[monthly_mask, "salary_mid_lpa"] = monthly_salary_lpa


### 7.2 Annual salaries

In [12]:
annual_mask = (
    ~monthly_mask
    & df_clean["salary"].notna()
    & ~df_clean["salary"].isin(["Not disclosed", "Unpaid"])
)

df_clean.loc[annual_mask, "salary_min_lpa"] = (
    df_clean.loc[annual_mask, "minimumSalary"] / 100000
)

df_clean.loc[annual_mask, "salary_max_lpa"] = (
    df_clean.loc[annual_mask, "maximumSalary"] / 100000
)

df_clean.loc[annual_mask, "salary_mid_lpa"] = (
    df_clean.loc[annual_mask, "salary_min_lpa"]
    + df_clean.loc[annual_mask, "salary_max_lpa"]
) / 2


### 7.3 Handle undisclosed and unpaid salaries

In [13]:
not_disclosed = df_clean["salary"] == "Not disclosed"

df_clean.loc[
    not_disclosed,
    ["salary_min_lpa", "salary_max_lpa", "salary_mid_lpa"]
] = np.nan

unpaid = df_clean["salary"] == "Unpaid"

df_clean.loc[
    unpaid,
    ["salary_min_lpa", "salary_max_lpa", "salary_mid_lpa"]
] = 0


### 7.4 Validate salary cleaning

In [14]:
print(df_clean[[
    "salary",
    "salary_min_lpa",
    "salary_max_lpa",
    "salary_mid_lpa"
]].head(20))


                 salary  salary_min_lpa  salary_max_lpa  salary_mid_lpa
0           2-4 Lacs PA             2.0            4.00           3.000
1           3-5 Lacs PA             3.0            5.00           4.000
2         Not disclosed             NaN             NaN             NaN
3      70,000-2 Lacs PA             0.7            2.00           1.350
4          8-18 Lacs PA             8.0           18.00          13.000
5         3.5-6 Lacs PA             3.5            6.00           4.750
6         Not disclosed             NaN             NaN             NaN
7         Not disclosed             NaN             NaN             NaN
8         Not disclosed             NaN             NaN             NaN
9         Not disclosed             NaN             NaN             NaN
10  50,000-1.75 Lacs PA             0.5            1.75           1.125
11        Not disclosed             NaN             NaN             NaN
12   50,000-2.5 Lacs PA             0.5            2.50         

In [15]:
invalid_salary_ranges = df_clean[
    df_clean["salary_min_lpa"] > df_clean["salary_max_lpa"]
]

print("Invalid salary ranges:", len(invalid_salary_ranges))


Invalid salary ranges: 0


## 8. Validate experience data

The dataset already contains numeric minimum and maximum experience fields, so we keep those fields instead of creating extra experience columns.


In [16]:
df_clean[
    ["experience", "minimumExperience", "maximumExperience"]
].head(20)


,experience,minimumExperience,maximumExperience
0,2-4 Yrs,2.0,4.0
1,6-11 Yrs,6.0,11.0
2,12-18 Yrs,12.0,18.0
3,0-3 Yrs,0.0,3.0
4,5-10 Yrs,5.0,10.0
5,10-15 Yrs,10.0,15.0
6,3-5 Yrs,3.0,5.0
7,2-3 Yrs,2.0,3.0
8,2-4 Yrs,2.0,4.0
9,4-14 Yrs,4.0,14.0


In [17]:
invalid_experience = df_clean[
    df_clean["minimumExperience"] > df_clean["maximumExperience"]
]

print("Invalid experience ranges:", len(invalid_experience))


Invalid experience ranges: 0


## 9. Prepare skills

`tagsAndSkills` contains comma-separated values. For now, we only split the values and remove extra spaces. Detailed skill normalization and analysis will be done later in SQL.


In [18]:
df_clean["skills_clean"] = df_clean["tagsAndSkills"].str.lower()
df_clean["skills_clean"] = df_clean["skills_clean"].str.split(",")


In [19]:
df_clean["skills_clean"] = df_clean["skills_clean"].apply(
    lambda skills: [skill.strip() for skill in skills]
    if isinstance(skills, list)
    else skills
)

df_clean["skills_clean"].head()


0    [communication, manpower, staffing, convincing...
1    [safety officer activities, fire protection, f...
2    [performance marketing, user acquisition, grow...
3    [fluent english, spoken english, good english ...
4    [product marketing, cns, product management, n...
Name: skills_clean, dtype: object

## 10. Identify Data Analyst jobs

For this project, the core Data Analyst market is defined as jobs whose title contains the phrase **Data Analyst**.


In [20]:
core_data_analyst = df_clean[
    df_clean["title"].str.contains(
        r"\bdata\s+analyst\b",
        case=False,
        na=False,
        regex=True
    )
].copy()

print("Data Analyst jobs:", len(core_data_analyst))


Data Analyst jobs: 267


### Check selected titles

In [21]:
core_data_analyst["title"].value_counts().head(50)


title
Data Analyst                                                              79
Senior Data Analyst                                                       18
Clinical Data Analyst and Programmer -  SAS+Veeva                          6
Client Data Analyst                                                        5
R & SAS Data Analyst                                                       4
Business Data Analyst                                                      4
Clinical Data Analyst: R and SAS Programming Specialist\n                  3
Lead Data Analyst                                                          3
Lead, Data Analyst                                                         3
Data Analyst - L4                                                          3
Data Analyst - L3                                                          3
Master Data Analyst                                                        2
Data Analyst II                                                       

### Check locations

In [22]:
core_data_analyst["location"].value_counts().head(20)


location
Bengaluru                         48
Hyderabad                         30
Gurugram                          20
Mumbai                            19
Pune                              18
Chennai                            9
Remote                             7
Hybrid - Hyderabad                 6
Hybrid - Bengaluru                 6
Ahmedabad                          5
Noida                              5
Vadodara                           4
Navi Mumbai                        3
Mumbai Suburban                    3
Kolkata                            3
Hybrid - Pune                      3
Hybrid - Mumbai(Vikhroli)          3
Mumbai(Powai)                      2
Hyderabad, Gurugram, Bengaluru     2
Coimbatore                         2
Name: count, dtype: int64

### Check experience requirements

In [23]:
core_data_analyst["experience"].value_counts().head(20)


experience
5-10 Yrs    24
3-8 Yrs     18
4-9 Yrs     16
5-8 Yrs     15
3-5 Yrs     15
1-3 Yrs     11
7-12 Yrs    11
4-8 Yrs     10
2-4 Yrs     10
2-6 Yrs      9
2-5 Yrs      8
1-4 Yrs      8
4-6 Yrs      8
3-6 Yrs      7
8-13 Yrs     7
6-11 Yrs     7
2-7 Yrs      7
6-10 Yrs     6
0-1 Yrs      6
1-5 Yrs      5
Name: count, dtype: int64

## 11. Create the final dataset for SQL

We keep the original fields needed for analysis and the three standardized salary fields.


In [24]:
final_columns = [
    "jobId",
    "title",
    "companyName",
    "location",
    "jobUploaded",
    "currency",
    "salary",
    "salary_min_lpa",
    "salary_max_lpa",
    "salary_mid_lpa",
    "minimumExperience",
    "maximumExperience",
    "tagsAndSkills",
    "ReviewsCount",
    "AggregateRating"
]

final_data = core_data_analyst[final_columns].copy()

print("Final shape:", final_data.shape)


Final shape: (267, 15)


## 12. Final validation

In [25]:
print("Final rows:", len(final_data))
print("Final columns:", len(final_data.columns))

final_data.info()


Final rows: 267
Final columns: 15
<class 'pandas.core.frame.DataFrame'>
Index: 267 entries, 883 to 97706
Data columns (total 15 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   jobId              267 non-null    int64  
 1   title              267 non-null    object 
 2   companyName        267 non-null    object 
 3   location           267 non-null    object 
 4   jobUploaded        267 non-null    object 
 5   currency           267 non-null    object 
 6   salary             267 non-null    object 
 7   salary_min_lpa     61 non-null     float64
 8   salary_max_lpa     61 non-null     float64
 9   salary_mid_lpa     61 non-null     float64
 10  minimumExperience  265 non-null    float64
 11  maximumExperience  265 non-null    float64
 12  tagsAndSkills      265 non-null    object 
 13  ReviewsCount       170 non-null    float64
 14  AggregateRating    170 non-null    float64
dtypes: float64(7), int64(1), object(7)
memory

## 13. Save the cleaned Data Analyst dataset

In [30]:
output_path = "data_analyst_jobs_cleaned.csv"
final_data.to_csv(output_path, index=False)
print("Saved:", output_path)


Saved: data_analyst_jobs_cleaned.csv


In [31]:
len(core_data_analyst)

267

In [32]:
core_data_analyst["jobId"].nunique()

267

In [33]:
core_data_analyst[core_data_analyst["jobId"].duplicated(keep=False)]

,title,jobId,currency,jobUploaded,companyName,tagsAndSkills,experience,salary,location,companyId,...,AggregateRating,jobDescription,minimumSalary,maximumSalary,minimumExperience,maximumExperience,salary_min_lpa,salary_max_lpa,salary_mid_lpa,skills_clean
